# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kratosontren/flyrank-ml-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from datasets import load_dataset
import pandas as pd
from itertools import islice

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

sample = pd.DataFrame(list(islice(daily, 5000)))

sample.head()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


# 0. Signal Check 1

## Signal
Average Search Position (gsc_avg_position)

## Why this signal?

This signal is related to FlyRank's ranking and CTR optimization logic.

## Hypothesis

Pages with poorer average search positions generally receive fewer clicks and may benefit from content improvements.

## Verdict

(To be filled after viewing the bucket table.)

In [7]:
import pandas as pd

position_bucket = (
    sample.groupby(
        pd.cut(
            sample["gsc_avg_position"],
            bins=[0,5,10,20,50,100]
        )
    )
    .agg(
        n=("gsc_avg_position","count"),
        avg_clicks=("gsc_clicks","mean"),
        avg_impressions=("gsc_impressions","mean")
    )
)

display(position_bucket)

print("Total observations:", len(sample))

/tmp/ipykernel_4596/225061502.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sample.groupby(


,n,avg_clicks,avg_impressions
gsc_avg_position,,,
"(0, 5]",488,0.327869,17.979508
"(5, 10]",1345,0.139777,10.276580
"(10, 20]",919,0.121872,12.119695
"(20, 50]",1446,0.044952,12.040802
"(50, 100]",782,0.002558,7.975703


Total observations: 5000


# 0. Signal Check 2

## Signal

Search Impressions (gsc_impressions)

## Why this signal?

This signal supports FlyRank's Quick-Win logic.

Pages with higher impressions represent larger optimization opportunities.

## Hypothesis

Higher impression pages deserve higher review priority.

## Verdict

(To be filled after viewing the bucket table.)

In [8]:
impression_bucket = (
    sample.groupby(
        pd.qcut(
            sample["gsc_impressions"],
            q=5,
            duplicates="drop"
        )
    )
    .agg(
        n=("gsc_impressions","count"),
        avg_clicks=("gsc_clicks","mean"),
        avg_position=("gsc_avg_position","mean")
    )
)

display(impression_bucket)

print("Total observations:", len(sample))

/tmp/ipykernel_4596/3022722913.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sample.groupby(


,n,avg_clicks,avg_position
gsc_impressions,,,
"(0.999, 2.0]",1363,0.013940,31.542920
"(2.0, 4.0]",713,0.054698,24.885928
"(4.0, 8.0]",947,0.070750,23.268223
"(8.0, 17.0]",1002,0.118762,22.823796
"(17.0, 424.0]",975,0.291282,23.538337


Total observations: 5000


# Signal Audit Summary

Signal 1 (Average Position)

Verdict:
CONFIRMED

Reason:
The bucket table shows a directional relationship between poorer average position and click performance, supporting its use in the baseline rule.

---

Signal 2 (Search Impressions)

Verdict:
CONFIRMED

Reason:
Pages with higher impressions represent larger opportunities for improvement and are suitable candidates for prioritization.

These findings are observational and should be interpreted as decision-support evidence rather than causal proof.


## My Rule

The baseline rule identifies pages that may benefit from content refresh by combining three historical signals:

1. High search impressions
2. Low click performance relative to visibility
3. Poor average search position

Pages with high impressions but relatively few clicks and weaker average positions receive higher scores because improving these pages could have a larger impact.

This rule is intended as a decision-support baseline rather than an automated decision system.

### Reason Codes

REFRESH_HIGH_IMPRESSIONS

REFRESH_LOW_CLICKS

REFRESH_POOR_POSITION

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

score = (
    sample["gsc_impressions"] / 100
    + sample["gsc_avg_position"]
    - sample["gsc_clicks"] / 10
)

sample["baseline_score"] = score

conditions = [
    sample["gsc_impressions"] > sample["gsc_impressions"].median(),
    sample["gsc_clicks"] < sample["gsc_clicks"].median(),
    sample["gsc_avg_position"] > sample["gsc_avg_position"].median()
]

choices = [
    "REFRESH_HIGH_IMPRESSIONS",
    "REFRESH_LOW_CLICKS",
    "REFRESH_POOR_POSITION"
]

sample["reason_code"] = np.select(
    conditions,
    choices,
    default="REVIEW"
)

sample["action"] = "REFRESH_CONTENT"

sample[
    [
        "baseline_score",
        "reason_code",
        "action"
    ]
].head()

,baseline_score,reason_code,action
0,4.133333,REFRESH_HIGH_IMPRESSIONS,REFRESH_CONTENT
1,71.650000,REFRESH_POOR_POSITION,REFRESH_CONTENT
2,34.010000,REFRESH_POOR_POSITION,REFRESH_CONTENT
3,23.393333,REFRESH_POOR_POSITION,REFRESH_CONTENT
4,17.850000,REFRESH_POOR_POSITION,REFRESH_CONTENT


## 2. Build the ranked queue (writes the CSV)

The baseline score is computed from historical search signals only.

No future outcomes, labels, or product flags are used.

Higher scores indicate pages that should be reviewed first for potential content refresh.

The ranked queue is written to:

work/outputs/baseline_action_score.csv

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

ranked = sample.sort_values(
    "baseline_score",
    ascending=False
)

output = Path("work/outputs")
output.mkdir(
    parents=True,
    exist_ok=True
)

ranked.to_csv(
    output / "baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")

ranked.head(10)

CSV written successfully.


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,baseline_score,reason_code,action
4286,2025-02-11,client_73cda7b4e4f265ea,content_6ac06aec2173eacb,True,True,True,False,1,0,127,...,0,0,0,0,0,0,0,127.01,REFRESH_POOR_POSITION,REFRESH_CONTENT
4000,2025-02-11,client_73cda7b4e4f265ea,content_516bb6f195d4ef04,True,True,True,False,1,0,117,...,0,0,0,0,0,0,0,117.01,REFRESH_POOR_POSITION,REFRESH_CONTENT
1071,2025-01-30,client_ff644d8251367cbb,content_0a467f792733701d,True,True,True,False,1,0,101,...,0,0,0,0,0,0,0,101.01,REFRESH_POOR_POSITION,REFRESH_CONTENT
204,2025-01-27,client_ff644d8251367cbb,content_3c2e782d3c81c503,True,True,True,False,1,0,101,...,0,0,0,0,0,0,0,101.01,REFRESH_POOR_POSITION,REFRESH_CONTENT
259,2025-01-27,client_ff644d8251367cbb,content_1d4c3551a46e2967,True,True,True,False,1,0,100,...,0,0,0,0,0,0,0,100.01,REFRESH_POOR_POSITION,REFRESH_CONTENT
501,2025-01-28,client_ff644d8251367cbb,content_b241043c0bb3c017,True,True,True,False,1,0,100,...,0,0,0,0,0,0,0,100.01,REFRESH_POOR_POSITION,REFRESH_CONTENT
578,2025-01-28,client_ff644d8251367cbb,content_9884db00882fe43f,True,True,True,False,2,0,199,...,0,0,0,0,0,0,0,99.52,REFRESH_POOR_POSITION,REFRESH_CONTENT
247,2025-01-27,client_ff644d8251367cbb,content_af1ba3aaa816fe60,True,True,True,False,2,0,198,...,0,0,0,0,0,0,0,99.02,REFRESH_POOR_POSITION,REFRESH_CONTENT
505,2025-01-28,client_ff644d8251367cbb,content_d9f15dd64498bcf6,True,True,True,False,1,0,99,...,0,0,0,0,0,0,0,99.01,REFRESH_POOR_POSITION,REFRESH_CONTENT
1044,2025-01-30,client_ff644d8251367cbb,content_5e38ee17ab85b2be,True,True,True,False,1,0,99,...,0,0,0,0,0,0,0,99.01,REFRESH_POOR_POSITION,REFRESH_CONTENT


## 3. Top-20 review

## Top-20 Review

Each row below represents a page recommended for review.

For every recommendation:

- Action: REFRESH_CONTENT
- Reason code: Generated by the baseline rule
- Confidence: Moderate
- What would make it wrong?

Possible explanations include seasonal search demand, temporary ranking fluctuations, recent content updates, or incomplete analytics coverage.

These recommendations should therefore be reviewed by a content owner before any action is taken.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked.head(20)

review = top20[
    [
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

review

,content_hash_id,baseline_score,reason_code,action
4286,content_6ac06aec2173eacb,127.010000,REFRESH_POOR_POSITION,REFRESH_CONTENT
4000,content_516bb6f195d4ef04,117.010000,REFRESH_POOR_POSITION,REFRESH_CONTENT
1071,content_0a467f792733701d,101.010000,REFRESH_POOR_POSITION,REFRESH_CONTENT
204,content_3c2e782d3c81c503,101.010000,REFRESH_POOR_POSITION,REFRESH_CONTENT
259,content_1d4c3551a46e2967,100.010000,REFRESH_POOR_POSITION,REFRESH_CONTENT
501,content_b241043c0bb3c017,100.010000,REFRESH_POOR_POSITION,REFRESH_CONTENT
578,content_9884db00882fe43f,99.520000,REFRESH_POOR_POSITION,REFRESH_CONTENT
247,content_af1ba3aaa816fe60,99.020000,REFRESH_POOR_POSITION,REFRESH_CONTENT
505,content_d9f15dd64498bcf6,99.010000,REFRESH_POOR_POSITION,REFRESH_CONTENT
1044,content_5e38ee17ab85b2be,99.010000,REFRESH_POOR_POSITION,REFRESH_CONTENT


## 4. Weak picks + leakage check
## Weak Picks

Some recommendations may be weak because:

- Search demand may be seasonal.
- Ranking changes may be temporary.
- Some pages may have incomplete Search Console or GA4 coverage.
- High impressions alone do not necessarily indicate poor content quality.

## Leakage Check

The baseline score uses only historical observations.

No future performance windows, labels, or product-generated flags were used.

The score is intended for prioritization and decision-support rather than prediction of future outcomes.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows with GSC available:")
print(sample["gsc_data_available"].sum())

print()

print("Rows with GA4 available:")
print(sample["ga4_data_available"].sum())

print()

print("Future labels used: NO")
print("Product flags used: NO")

Rows with GSC available:
5000

Rows with GA4 available:
0

Future labels used: NO
Product flags used: NO


## Self-Check

✅ Every section above contains markdown explanations and supporting code.

✅ The notebook executes successfully from top to bottom.

✅ No client names, URLs, or private search queries are included.

✅ Claims use careful language such as observed, measured, directional, and decision-support.

✅ The notebook has been committed under:

work/notebooks/w04_baseline_score.ipynb

✅ The ranked queue is written to:

work/outputs/baseline_action_score.csv